# 📊 Notebook 01 – Exploración y Limpieza del Dataset
**Smart Diet Planner | CC3045 – Inteligencia Artificial**

Este notebook explora los archivos descargados del USDA FoodData Central (SR Legacy),
los combina en un solo dataset limpio y lo guarda en `data/processed/foods_clean.csv`.

**Archivos de entrada:**
- `data/raw/food.csv` → nombres de alimentos
- `data/raw/nutrient.csv` → diccionario de nutrientes
- `data/raw/food_nutrient.csv` → valores nutricionales por alimento

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Rutas
RAW = '../data/raw/'
PROCESSED = '../data/processed/'
os.makedirs(PROCESSED, exist_ok=True)

print('✅ Imports listos')
print(f'📁 raw:       {os.path.abspath(RAW)}')
print(f'📁 processed: {os.path.abspath(PROCESSED)}')

## 2. Cargar los 3 archivos base

In [ ]:
# Nombres de alimentos
food = pd.read_csv(RAW + 'food.csv')
print(f'food.csv         → {food.shape[0]:,} filas, {food.shape[1]} columnas')
food.head(3)

In [ ]:
# Diccionario de nutrientes
nutrient = pd.read_csv(RAW + 'nutrient.csv')
print(f'nutrient.csv     → {nutrient.shape[0]:,} filas, {nutrient.shape[1]} columnas')
nutrient.head(10)

In [ ]:
# Valores nutricionales (archivo más grande)
food_nutrient = pd.read_csv(RAW + 'food_nutrient.csv')
print(f'food_nutrient.csv → {food_nutrient.shape[0]:,} filas, {food_nutrient.shape[1]} columnas')
food_nutrient.head(3)

## 3. Identificar los nutrientes que nos interesan

In [ ]:
# Ver todos los nutrientes disponibles
print('Nutrientes disponibles en el dataset:')
print(nutrient[['id', 'name', 'unit_name']].to_string())

In [ ]:
# Los nutrientes que necesita nuestro proyecto
NUTRIENTES_OBJETIVO = {
    1008: 'calories',    # Energía (kcal)
    1003: 'protein',     # Proteína (g)
    1005: 'carbs',       # Carbohidratos (g)
    1004: 'fat',         # Grasa total (g)
    1079: 'fiber',       # Fibra (g)
    2000: 'sugar',       # Azúcares (g)
}

# Verificar cuáles existen en el dataset
encontrados = nutrient[nutrient['id'].isin(NUTRIENTES_OBJETIVO.keys())][['id', 'name', 'unit_name']]
print('Nutrientes encontrados en el dataset:')
print(encontrados)

## 4. Filtrar y pivotar food_nutrient

In [ ]:
# Quedarse solo con los nutrientes de interés
fn_filtrado = food_nutrient[food_nutrient['nutrient_id'].isin(NUTRIENTES_OBJETIVO.keys())].copy()
print(f'Filas después del filtro: {fn_filtrado.shape[0]:,}')

# Renombrar nutrient_id con nombre legible
fn_filtrado['nutrient_name'] = fn_filtrado['nutrient_id'].map(NUTRIENTES_OBJETIVO)

fn_filtrado[['fdc_id', 'nutrient_id', 'nutrient_name', 'amount']].head(10)

In [ ]:
# Pivotar: cada fila = un alimento, cada columna = un nutriente
pivote = fn_filtrado.pivot_table(
    index='fdc_id',
    columns='nutrient_name',
    values='amount',
    aggfunc='first'
).reset_index()

print(f'Dataset pivoteado: {pivote.shape[0]:,} alimentos x {pivote.shape[1]} columnas')
pivote.head()

## 5. Unir con nombres de alimentos

In [ ]:
# Columnas útiles de food.csv
print('Columnas en food.csv:', food.columns.tolist())
food_base = food[['fdc_id', 'description']].copy()
food_base.head()

In [ ]:
# Merge: unir nombre del alimento con sus nutrientes
df = food_base.merge(pivote, on='fdc_id', how='inner')
print(f'Dataset final antes de limpiar: {df.shape[0]:,} alimentos')
df.head()

## 6. Limpieza de datos

In [ ]:
# Ver cuántos valores nulos hay por columna
print('Valores nulos por columna:')
print(df.isnull().sum())
print(f'\nTotal de filas: {len(df):,}')

In [ ]:
# Eliminar filas sin calorías (son el dato más importante)
df = df.dropna(subset=['calories'])
print(f'Después de eliminar sin calorías: {len(df):,} alimentos')

# Rellenar nulos restantes con 0
cols_nutrientes = ['protein', 'carbs', 'fat', 'fiber', 'sugar']
df[cols_nutrientes] = df[cols_nutrientes].fillna(0)

# Eliminar duplicados por nombre
df = df.drop_duplicates(subset='description')
print(f'Después de eliminar duplicados: {len(df):,} alimentos')

# Eliminar alimentos con calorías = 0 (no aportan info)
df = df[df['calories'] > 0]
print(f'Después de eliminar calorías = 0: {len(df):,} alimentos')

# Eliminar valores absurdos (más de 1000 kcal por 100g)
df = df[df['calories'] <= 1000]
print(f'Después de eliminar outliers: {len(df):,} alimentos')

In [ ]:
# Limpiar nombre del alimento
df['description'] = df['description'].str.strip().str.title()

# Renombrar columnas al español para el proyecto
df = df.rename(columns={'description': 'nombre'})

# Redondear valores numéricos a 2 decimales
for col in cols_nutrientes + ['calories']:
    df[col] = df[col].round(2)

# Resetear índice
df = df.reset_index(drop=True)

print('✅ Limpieza completada')
df.head(10)

## 7. Estadísticas descriptivas

In [ ]:
print('=== RESUMEN DEL DATASET LIMPIO ===')
print(f'Total de alimentos: {len(df):,}')
print(f'Columnas: {df.columns.tolist()}')
print()
df[['calories', 'protein', 'carbs', 'fat', 'fiber', 'sugar']].describe().round(2)

## 8. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribución de Nutrientes en el Dataset', fontsize=16, fontweight='bold')

cols_plot = ['calories', 'protein', 'carbs', 'fat', 'fiber', 'sugar']
colores = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c', '#9b59b6', '#f39c12']

for ax, col, color in zip(axes.flatten(), cols_plot, colores):
    ax.hist(df[col], bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(col.capitalize(), fontweight='bold')
    ax.set_xlabel('Por 100g')
    ax.set_ylabel('Frecuencia')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(PROCESSED + 'distribucion_nutrientes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada')

In [ ]:
# Top 15 alimentos más calóricos
top_cal = df.nlargest(15, 'calories')[['nombre', 'calories', 'fat', 'protein']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_cal['nombre'], top_cal['calories'], color='#e74c3c', alpha=0.85)
ax.set_xlabel('Calorías (kcal por 100g)', fontweight='bold')
ax.set_title('Top 15 Alimentos Más Calóricos', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(PROCESSED + 'top_caloricos.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top 15 alimentos más proteicos
top_prot = df.nlargest(15, 'protein')[['nombre', 'protein', 'calories']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_prot['nombre'], top_prot['protein'], color='#3498db', alpha=0.85)
ax.set_xlabel('Proteína (g por 100g)', fontweight='bold')
ax.set_title('Top 15 Alimentos Más Proteicos', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(PROCESSED + 'top_proteicos.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Guardar dataset limpio

In [ ]:
output_path = PROCESSED + 'foods_clean.csv'
df.to_csv(output_path, index=False)

print('✅ Dataset limpio guardado exitosamente')
print(f'📁 Ruta: {os.path.abspath(output_path)}')
print(f'📊 Alimentos: {len(df):,}')
print(f'📋 Columnas: {df.columns.tolist()}')
print()
print('Muestra final:')
df.sample(5)